In [25]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm

In [26]:
# --- 1단계: 데이터 준비 및 전처리 ---
print("1. 데이터 준비 및 전처리 시작...")


1. 데이터 준비 및 전처리 시작...


In [27]:
KINDS_PATH = '../../data/interim/news/deep_search_news.csv'
STOPWORD_PATH = '../../data/raw/news/stopwords-ko.txt'

POSITIVE_PATH = '../../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../../data/raw/news/sentiment/natural.txt'
# 뉴스 데이터 읽기
df = pd.read_csv(KINDS_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")
print(f"수집된 기사의 길이 : {len(df)}")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...
수집된 기사의 길이 : 37437


In [28]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    if not isinstance(text, str):
        return []
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['오후', '대전시', '지족동', '아파트', '지하', '주차장', '주차', '승용차', '차량', '아파트', '경비원', '진화']


In [29]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [30]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...   
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...   
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...   
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...   
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...   

                                   textrank_keywords  
0  [아파트, 대전시, 경비원, 지족동, 주차, 주차장, 승용차, 지하, 차량, 오후,...  
1  [중공업, 지난달, 동구, 현대, 평균, 울산시, 기록, 본격, 중단, 발표, 가동...  
2  [사업, 조성, 부지, 거제, 고현항, 개발, 녹지, 주거, 공원, 문화, 부족, ...  
3  [매매, 산리, 전원, 삼익, 비치, 생면, 주택, 남천동, 욕실, 주거, 지역, ...  
4  [주택, 시장, 부담, 종부, 인상, 반사, 건물, 상업, 소형, 관망세, 작용, ...  


In [31]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...         0.000000
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...        -0.075000
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...         0.090909
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...         0.240000
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...        -0.187500


In [32]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [33]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [34]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.to_period('M')


In [35]:
print("\n월별 감성 지수:")
print(monthly_sentiment)
monthly_sentiment


월별 감성 지수:
      month  sentiment_score
0   2018-07         0.016983
1   2018-08         0.035666
2   2018-09         0.013844
3   2018-10         0.023940
4   2018-11         0.004340
..      ...              ...
82  2025-05         0.045637
83  2025-06         0.003236
84  2025-07        -0.007471
85  2025-08         0.010208
86  2025-09         0.007458

[87 rows x 2 columns]


,month,sentiment_score
0,2018-07,0.016983
1,2018-08,0.035666
2,2018-09,0.013844
3,2018-10,0.023940
4,2018-11,0.004340
...,...,...
82,2025-05,0.045637
83,2025-06,0.003236
84,2025-07,-0.007471
85,2025-08,0.010208


In [36]:
SALE_PATH = '../../data/interim/apt/gang_nam_apt_with_long_lat.csv'
sale = pd.read_csv(SALE_PATH)
len(sale)

21561

In [37]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.to_period('M')
drop_col = ['단지명','도로명','경도','위도']

In [41]:
sale.columns

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', '계약일자', 'month'], dtype='object')

In [42]:
# sale.drop(drop_col, axis=1, inplace=True)

In [43]:
merged_df = pd.merge(sale, monthly_sentiment, on='month', how='left')
merged_df.drop('month', axis=1, inplace=True)
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,계약일자,sentiment_score
0,84.73,5,1994,7.399156,24,2018-07-01,0.016983
1,169.19,4,2003,6.753467,15,2018-07-01,0.016983
2,84.43,10,1980,7.618163,38,2018-07-02,0.016983
3,76.79,4,1979,7.570627,39,2018-07-02,0.016983
4,162.51,1,1999,6.898420,19,2018-07-02,0.016983


In [44]:
len(merged_df)

21561

In [45]:
merged_df.to_csv('../../data/interim/gang_nam_sendimental_score_with_sale.csv',index=False)